In [1]:
%%time
import warnings
import optuna
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score, KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error, make_scorer
from catboost import CatBoostRegressor, Pool, EShapCalcType, EFeaturesSelectionAlgorithm
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import ExtraTreesRegressor

warnings.filterwarnings('ignore')

Wall time: 43.1 s


In [2]:
df=pd.read_csv("training_dataset.csv")
df.drop(["region__peak_sales_dollar_amt_per_hour_v2","region__peak_returns_dollar_amt_per_hour_v2","Unnamed: 0","observation_id","observation_timestamp"], axis=1, inplace=True)
df.head()

,hour_of_day,register__sales_dollar_amt_this_hour,register__payment_types_accepted,register__peak_sales_dollar_amt_per_hour,register__sales_dollar_amt_last_hour,register__sales_quantity_last_hour,register__sales_quantity_rescanned_frac,register__sales_payments_declined_frac,register__peak_returns_dollar_amt_per_hour,register__returns_dollar_amt_last_hour,...,region__mean_service_time_per_customer,region__stdev_service_time_per_customer,region__sales_dollar_amt_last_hour,region__returns_dollar_amt_last_hour,region__nighttime_open_registers,region__nighttime_service_time_per_customer,region__nighttime_sales_amt_per_hour,region__nighttime_returns_amt_per_hour,region__peak_sales_dollar_amt_per_hour,region__peak_returns_dollar_amt_per_hour
0,15,347.29,Cash+Credit,-0.7383,-0.1270,-0.1993,-0.8299,-0.1247,-0.5721,-0.2582,...,-0.9255,-0.3891,-0.6920,-0.4605,-0.5180,-1.0062,-0.6462,-0.6030,-0.4773,-1.7951
1,21,361.59,Cash+Credit,0.6483,-0.0362,-0.0777,-0.7395,-0.1135,-0.5251,-0.2157,...,0.7815,0.0663,-0.6531,-0.4434,-0.6498,0.9031,-0.6493,-0.6106,0.4998,0.8939
2,11,850.73,Cash+Credit,-0.4950,-0.1268,-0.1974,1.3139,0.1075,-0.5251,-0.2579,...,-1.2559,-0.4826,-0.6120,-0.1786,-0.6040,-1.4229,-0.6456,-0.6037,1.9337,2.4046
3,10,1175.69,Cash+Credit,-0.5594,-0.1270,-0.1991,-0.8299,-0.1247,-0.5251,-0.2582,...,-1.2280,-0.4809,-0.5472,0.1246,-0.5925,-1.4097,-0.6478,-0.6079,0.1288,-0.4983
4,14,3204.53,Cash+Credit,0.5693,-0.1221,-0.1632,-0.7071,-0.1247,0.7604,-0.2565,...,0.2442,-0.3021,-0.3838,0.1996,-0.5696,1.1231,-0.6475,-0.6032,1.0050,1.5660


In [3]:
def regression_accuracy(y_meas, y_pred, max_error=20, error_type='relative'):
    '''Compares predicted & measured values, returning the percentage of predictions
       that are within a set error limit. This error limit can be an absolute value
       or a relative percentage'''

    # OPTION 1: Relative percentage
    if ( error_type == 'relative' ):
        mask = 100.0 * abs((y_pred - y_meas) / y_meas) < max_error

    # OPTION 2: Absolute value
    if ( error_type == 'absolute' ):
        mask = abs(y_pred - y_meas) < max_error

    accuracy = sum(mask) / len(mask)

    return accuracy

class AccuracyMetric(object):
    def get_final_error(self, error, weight):
        return error / (weight + 1e-38)

    def is_max_optimal(self):
        return True

    def evaluate(self, approxes, target, weight):
        best_class = np.argmax(approxes, axis=0)

        accuracy_sum = 0
        weight_sum = 0

        for i in range(len(target)):
            w = 1.0 if weight is None else weight[i]
            weight_sum += w
            accuracy_sum += w * (best_class[i] == target[i])

        return accuracy_sum, weight_sum


scorer=make_scorer(regression_accuracy)

In [4]:
def imputation(df):
    return df

In [5]:
def encodage(df):
    return df

In [6]:
def pre_processing(df):
    #df=imputation(df)
    df=encodage(df)

    if "register__sales_dollar_amt_this_hour" in df.columns.values:
        X = df.drop("register__sales_dollar_amt_this_hour", axis=1)
        y = df.register__sales_dollar_amt_this_hour
        return X,y
    else:
        X = df
        return X

In [7]:
df.store__is_sufficiently_staffed=LabelEncoder().fit_transform(df.store__is_sufficiently_staffed)

In [8]:
df.select_dtypes("object").columns

Index(['register__payment_types_accepted', 'store__type_code'], dtype='object')

In [9]:
df.register__payment_types_accepted=LabelEncoder().fit_transform(df.register__payment_types_accepted)

In [10]:
df.store__type_code=LabelEncoder().fit_transform(df.store__type_code)

In [11]:
with open("used_columns.txt","r") as file:
    used_columns=[i.replace("\n","") for i in file.readlines()]

# used_columns.extend(df.select_dtypes("object").columns)
# used_columns.append("store__type_code")
# df=df[used_columns]
train,test=train_test_split(df, test_size=0.2)
X_train,y_train=pre_processing(train)
X_test,y_test=pre_processing(test)

In [12]:
train_data=Pool(
    data=X_train,
    label=y_train,
    cat_features=['register__payment_types_accepted', 'store__type_code',"store__is_sufficiently_staffed"]
)
eval_data=Pool(
    data=X_test,
    label=y_test,
    cat_features=['register__payment_types_accepted', 'store__type_code',"store__is_sufficiently_staffed"]
)

In [13]:
model = CatBoostRegressor(iterations=1000, loss_function='RMSE',random_seed=0)
summary = model.select_features(
    train_data,
    eval_set=eval_data,
    features_for_select='0-56',
    num_features_to_select=30,
    steps=10,
    algorithm=EFeaturesSelectionAlgorithm.RecursiveByShapValues,
    shap_calc_type=EShapCalcType.Regular,
    train_final_model=True,
    logging_level='Silent',
    plot=True
)

MetricVisualizer(layout=Layout(align_self='stretch', height='500px'))

In [14]:
loss=summary["loss_graph"]["loss_values"]
eliminated_features=summary["eliminated_features_names"]
features=[]
for i,feature in enumerate(eliminated_features):
    if (loss[i]<loss[i+1]):
        features.append(feature)

In [15]:
used_columns=summary["selected_features_names"]
used_columns.extend(features)

In [16]:
#X=df[used_columns]
X=df.drop("register__sales_dollar_amt_this_hour",axis=1)
y=df.register__sales_dollar_amt_this_hour

In [17]:
cv=KFold(n_splits=5,shuffle=True,random_state=0)

In [18]:
model=CatBoostRegressor(iterations=1000,learning_rate=0.05,depth=8)

scores=[]
for i,j in enumerate(cv.split(X,y)):
    train=Pool(
        data=X.iloc[j[0]],
        label=y.iloc[j[0]],
    )
    eval=Pool(
        data=X.iloc[j[1]],
        label=y.iloc[j[1]]
    )
    model.fit(
        train,
        eval_set=eval,
        verbose=False
    )
    y_pred=model.predict(X.iloc[j[1]])
    score=regression_accuracy(y.iloc[j[1]], y_pred)
    scores.append(score)
    print(f"Fold {i}\t:{score}\n")

print(f"CV_SCORES_mean: {np.mean(scores)}")
print(f"CV_SCORES_std: {np.std(scores)}")

Fold 0	:0.8601134215500945

Fold 1	:0.8601134215500945

Fold 2	:0.8660183684494868

Fold 3	:0.8595353862776878

Fold 4	:0.8641274986493788

CV_SCORES_mean: 0.8619816192953484
CV_SCORES_std: 0.0026024804914898786


In [17]:
def objective(trial):
    params = {
        "subsample": trial.suggest_float("subsample", 0.5, 0.99),
        'od_wait': trial.suggest_int('od_wait', 10, 50, step=2),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.2, 0.99),
        "random_strength": trial.suggest_int("random_strength", 1, 10, step=1),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.5),
        "max_depth": trial.suggest_int("max_depth", 4, 15, step=1),
        "n_estimators": trial.suggest_int("n_estimators", 50, 1000, step=1),
        'learning_rate': trial.suggest_loguniform("learning_rate", 0.01, 0.2),
        }

    model=CatBoostRegressor(**params)

    scores=[]
    for i,j in enumerate(cv.split(X,y)):
        train=Pool(
            data=X.iloc[j[0]],
            label=y.iloc[j[0]],
        )
        eval=Pool(
            data=X.iloc[j[1]],
            label=y.iloc[j[1]]
        )
        model.fit(
            train,
            eval_set=eval,
            early_stopping_rounds=100,
            verbose=False
        )
        y_pred=model.predict(X.iloc[j[1]])
        score=regression_accuracy(y.iloc[j[1]], y_pred)
        scores.append(score)

        return np.mean(scores)


In [18]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=300, n_jobs=-1, timeout=40000)

[I 2022-09-16 09:26:11,181] A new study created in memory with name: no-name-87eab311-3c9b-4d30-b9a2-6429354c2a6f
Custom logger is already specified. Specify more than one logger at same time is not thread safe.Custom logger is already specified. Specify more than one logger at same time is not thread safe.Custom logger is already specified. Specify more than one logger at same time is not thread safe.[I 2022-09-16 09:37:02,675] Trial 2 finished with value: 0.8520118822576289 and parameters: {'subsample': 0.8986969752785168, 'od_wait': 34, 'colsample_bylevel': 0.5719621897578335, 'random_strength': 3, 'l2_leaf_reg': 2.7796239473370576, 'max_depth': 13, 'n_estimators': 819, 'learning_rate': 0.13817861119498334}. Best is trial 2 with value: 0.8520118822576289.
[I 2022-09-16 09:41:18,488] Trial 4 finished with value: 0.8449905482041588 and parameters: {'subsample': 0.6808260074313915, 'od_wait': 44, 'colsample_bylevel': 0.22536982319166968, 'random_strength': 4, 'l2_leaf_reg': 9.220696995

: 

In [188]:
optuna.visualization.plot_optimization_history(study)

In [189]:
study.best_params

{'subsample': 0.5345222398899366,
 'od_wait': 10,
 'colsample_bylevel': 0.6959863997197222,
 'random_strength': 2,
 'l2_leaf_reg': 2.7409700776028845,
 'max_depth': 11,
 'n_estimators': 454,
 'learning_rate': 0.04074589813699164}

In [190]:


# 5567.471491542595.

In [191]:
best_model=CatBoostRegressor(**best_params)
train=Pool(data=X,label=y)
best_model.fit(train)

0:	learn: 789.3634180	total: 224ms	remaining: 1m 41s
1:	learn: 762.1746821	total: 362ms	remaining: 1m 21s
2:	learn: 735.2369713	total: 541ms	remaining: 1m 21s
3:	learn: 710.0543386	total: 720ms	remaining: 1m 21s
4:	learn: 686.3700353	total: 896ms	remaining: 1m 20s
5:	learn: 663.8364252	total: 1.11s	remaining: 1m 22s
6:	learn: 641.2930284	total: 1.32s	remaining: 1m 24s
7:	learn: 620.4379414	total: 1.52s	remaining: 1m 24s
8:	learn: 599.7848484	total: 1.69s	remaining: 1m 23s
9:	learn: 580.4556538	total: 1.87s	remaining: 1m 22s
10:	learn: 561.9006336	total: 2.05s	remaining: 1m 22s
11:	learn: 544.0417089	total: 2.2s	remaining: 1m 21s
12:	learn: 526.6362632	total: 2.4s	remaining: 1m 21s
13:	learn: 510.1749322	total: 2.58s	remaining: 1m 21s
14:	learn: 494.2669512	total: 2.75s	remaining: 1m 20s
15:	learn: 478.8450422	total: 2.96s	remaining: 1m 21s
16:	learn: 464.3448688	total: 3.14s	remaining: 1m 20s
17:	learn: 450.0587288	total: 3.34s	remaining: 1m 20s
18:	learn: 436.9767441	total: 3.55s	rema

In [192]:
test_df=pd.read_csv("test_dataset.csv")
test_df.drop(["region__peak_sales_dollar_amt_per_hour_v2","region__peak_returns_dollar_amt_per_hour_v2"], axis=1, inplace=True)
test_df.head()

,observation_id,observation_timestamp,hour_of_day,register__payment_types_accepted,register__peak_sales_dollar_amt_per_hour,register__sales_dollar_amt_last_hour,register__sales_quantity_last_hour,register__sales_quantity_rescanned_frac,register__sales_payments_declined_frac,register__peak_returns_dollar_amt_per_hour,...,region__mean_service_time_per_customer,region__stdev_service_time_per_customer,region__sales_dollar_amt_last_hour,region__returns_dollar_amt_last_hour,region__nighttime_open_registers,region__nighttime_service_time_per_customer,region__nighttime_sales_amt_per_hour,region__nighttime_returns_amt_per_hour,region__peak_sales_dollar_amt_per_hour,region__peak_returns_dollar_amt_per_hour
0,3f483640-bc52-11ec-b736-8544dc068949,15/04/2022 00:23,0,Cash+Credit,0.5693,-0.1253,-0.1489,-0.8176,-0.1066,0.9103,...,-0.1320,0.3398,-0.6373,-0.3791,-0.5581,-0.7148,-0.6487,-0.6091,1.0020,0.8761
1,a88b0bb0-d2ae-11ec-bd0e-f5a7c7895456,13/05/2022 11:20,11,Cash+Credit,0.3084,-0.1270,-0.1986,1.6562,-0.1247,-0.5251,...,0.8413,0.1972,-0.6483,-0.4419,-0.6498,0.8867,-0.6493,-0.6106,-1.9779,-0.1311
2,90611780-d607-11ec-a5d8-675445c92326,17/05/2022 17:34,17,Cash+Credit,-0.6757,0.1925,0.2534,-0.7517,-0.1247,-0.5251,...,-0.3185,-0.4461,0.6712,0.3971,2.0094,0.0464,1.8275,2.0689,0.0332,-0.3489
3,a98380e0-e2d2-11ec-b28f-7f4d381958ee,03/06/2022 00:18,0,Cash+Credit,-0.1747,-0.1258,-0.1709,-0.3576,-0.1247,-0.5251,...,0.8032,0.1548,-0.6653,-0.4437,-0.6498,0.8937,-0.6493,-0.6106,0.4998,0.8939
4,608b8e60-e4f2-11ec-8115-d3258731976b,05/06/2022 17:10,17,Cash+Credit,0.6483,-0.0612,-0.0995,0.1060,-0.1247,-0.5251,...,0.9011,0.1806,-0.6495,-0.4418,-0.6498,0.9006,-0.6493,-0.6106,0.4998,0.8939


In [193]:
test_df=test_df[X.columns]
predictions = pd.DataFrame(best_model.predict(test_df), columns=["prediction"])
predictions.head()

,prediction
0,1827.595288
1,361.993511
2,1410.346209
3,244.876024
4,355.368458


In [185]:
predictions.to_csv("submit_38.csv", index=False)